# dl_30_build_gold

Runs `Files/sql/gold/*.sql` in filename order. `00_source_views.sql` is
the seam: every other file reads `sv_*` and nothing else, which is what
lets the same SQL be tested offline in DuckDB by `tests/test_gold.py`.

Gold is rebuilt in full every run. It is small, and recomputing it is
simpler and self-healing compared with incremental merge logic that
nobody can debug six months later.

In [ ]:
import sys
sys.path.insert(0, "/lakehouse/default/Files/lib")

LIB = "/lakehouse/default/Files"

import glob
import os
from fabric_common import split_sql_statements, log_run, new_batch_id, utc_now


def run_sql_folder(folder: str, batch_id: str, step: str) -> None:
    """Execute every .sql file in a folder, in FILENAME ORDER.

    Ordering lives in the numeric prefix and the logic lives in version-
    controlled SQL, so a transform can be reviewed in a pull request rather than
    clicked through in a dataflow.
    """
    paths = sorted(glob.glob(os.path.join(folder, "*.sql")))
    if not paths:
        raise RuntimeError(f"no .sql files found in {folder} - is Files/ uploaded?")
    for path in paths:
        with open(path, encoding="utf-8") as handle:
            statements = split_sql_statements(handle.read())
        for statement in statements:
            spark.sql(statement)
        print(f"  {os.path.basename(path):48s} {len(statements)} statement(s)")
        log_run(spark, batch_id, step, os.path.basename(path), len(statements))

In [ ]:
batch_id = new_batch_id()
print(f"batch {batch_id}")
run_sql_folder(f"{LIB}/sql/gold", batch_id, "build_gold")

In [ ]:
import json


def write_diag(name: str, payload: dict) -> None:
    """Structured diagnostics to Files/_diag/.

    Fabric's job API gives no per-cell detail - a failed notebook reports
    "Failed" and nothing else. Writing what happened to a file the deploy
    scripts can read back is the difference between debugging this and guessing.
    """
    os.makedirs("/lakehouse/default/Files/_diag", exist_ok=True)
    path = f"/lakehouse/default/Files/_diag/{name}.json"
    with open(path, "w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, default=str)
    print(f"diagnostics -> {path}")

tables = [row.tableName for row in spark.sql("SHOW TABLES").collect()
          if row.tableName.startswith(("dim_", "fct_", "meta_"))]
counts = {t: spark.table(t).count() for t in sorted(tables)}
for table, count in counts.items():
    print(f"  {table:32s} {count:8,d}")

# The semantic model infers its column types from these tables. A type that
# disagrees with the model makes Direct Lake drop the table SILENTLY, so the
# schema is captured here for deploy_model.py to generate TMDL from.
schema = {
    t: [{"name": f.name, "type": f.dataType.simpleString()} for f in spark.table(t).schema]
    for t in sorted(tables)
}
write_diag("gold_schema", {"batch_id": batch_id, "counts": counts, "schema": schema})